### Methods for reading HDF5 files and exporting to dictionary structure.

In [ ]:
import h5py
from matplotlib import colors
import matplotlib.pyplot as plt
import numpy as np
import os
import re

from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from IPython.display import display as ipydisplay

from scipy.signal import savgol_filter

# Threshold for peak-to-average ratio acceptance of image.
THRESHOLD = 10


def h5_to_dict(filename):
    """Read an HDF5 file into a nested dict.

    Structure:
        data[group_name]['attrs']        -> group attributes (metadata)
        data[group_name][dataset_name]   -> dict with 'data' (numpy array)
                                           and 'attrs' (dataset attributes)
    """
    def _read_group(grp):
        out = {'attrs': dict(grp.attrs)}
        for name, item in grp.items():
            if isinstance(item, h5py.Dataset):
                out[name] = {'data': item[()], 'attrs': dict(item.attrs)}
            elif isinstance(item, h5py.Group):
                out[name] = _read_group(item)
        return out

    with h5py.File(filename, 'r') as f:
        return {key: _read_group(f[key]) for key in f}


def data_from_h5_files(files):
    """Read multiple HDF5 files into a nested dict."""
    data = {}
    for filename in files:
        key = re.sub('pass', '',
                     os.path.basename(filename).split('_')[2].split('-')[0])
        data[key] = h5_to_dict(filename)
    return data


def files_in_directory(wdir, pattern):
    """List files in a directory matching a regex pattern."""
    rawfiles = os.listdir(wdir)
    return sorted([f"{wdir}/{f}" for f in rawfiles if re.match(pattern, f)])


def beam_visible(img, cx, cy, droi=4, threshold=THRESHOLD):
    """Detect if a beam is present based on the peak-to-average ratio.

    Args:
        img (np.array): The 2D image array to analyze.
        cx (int): The x-coordinate of the beam centroid.
        cy (int): The y-coordinate of the beam centroid.
        droi (int): The half-size of the region around the centroid used
            to calculate the peak intensity. Default is 4 pixels.
        threshold (float): The minimum peak-to-average ratio required to
            consider the beam visible. Default is 10.

    Returns:
        bool: True if the beam is considered visible, False otherwise.
    """
    roi_avg = np.mean(img[cy-droi:cy+droi, cx-droi:cx+droi])
    mean = np.mean(img)
    ratio_rtom = roi_avg / mean if mean != 0 else 0

    return ratio_rtom >= threshold


def beam_properties(dataset, dev_motor, droi=4, threshold=THRESHOLD):
    """Return properties of the beam profiles from the data dict.

    Args:
        dataset (dict): Nested dict containing the set of one full scan.
        dev_motor (str): The device and motor being scanned (e.g., 'sample.x').
        droi (int): The half-size of the region around the centroid used to
            determine if the beam is visible. Default is 4 pixels.
        threshold (float): The minimum peak-to-average ratio required to
            consider the beam visible. Default is 10.

    Returns:
        beam_propties : A dict mapping scan numbers to (cx, cy), (fx, fy)
            centroids and FWHMs.
        beam_images   : A dict mapping scan numbers to DVF images of the beam,
            only scans where the beam is visible are returned.
    """
    beam_propties = {}
    beam_images   = {}
    xval          = []
    for scan, scandata in dataset.items():
        # Extract scanning index and observable value.
        sc = int(scan.split('-')[-1])
        xval = float(scandata['attrs'].get(dev_motor)[0])

        # Get image data and calculate centroid.
        img = scandata['dvf_B1']['data']

        # Centroids.
        # cx = np.sum(img, axis=0).argmax()
        # cy = np.sum(img, axis=1).argmax()
        cx_sum = np.sum(img, axis=0)
        cy_sum = np.sum(img, axis=1)

        xsmooth = savgol_filter(cx_sum, window_length=21, polyorder=2)
        ysmooth = savgol_filter(cy_sum, window_length=21, polyorder=2)

        cx = np.argmax(xsmooth)
        cy = np.argmax(ysmooth)

        # DEBUG
        # xmaxraw = np.argmax(cx_sum)
        # ymaxraw = np.argmax(cy_sum)
        # print(f">>>>> Scan {sc}: Centroid raw at (cx={xmaxraw},"
        #       f" cy={ymaxraw}), ")
        # print(f">>>>> Scan {sc}: Centroid at (cx={cx}, cy={cy}), ")
        # # DEBUG

        # FWHMs
        x_profile = img[cy, :]
        y_profile = img[:, cx]

        fwhm_x = np.sum(x_profile > (x_profile.max() / 2))
        fwhm_y = np.sum(y_profile > (y_profile.max() / 2))

        # Do not register properties if there is no beam image.
        if not beam_visible(img, cx, cy, droi, threshold):
            continue

        # Dict is indexed by scan number.
        beam_propties[sc] = [xval, [cx, cy], [fwhm_x, fwhm_y]]
        beam_images[sc]   = img
 
    return beam_images, beam_propties


def beam_centroid(dataset, dev_motor, droi=4, threshold=THRESHOLD):
    """Return centroids of the beam profiles from the data dict.

    Args:
        dataset (dict): Nested dict containing the set of one full scan.
        dev_motor (str): The device and motor being scanned (e.g., 'sample.x').
        droi (int): The half-size of the region around the centroid used to
            determine if the beam is visible. Default is 4 pixels.
        threshold (float): The minimum peak-to-average ratio required to
            consider the beam visible. Default is 10.

    Returns:
        dict: A dict mapping scan numbers to (cx, cy) centroids.
    """
    centroids = {}
    xval      = []

    _, beam_propties = beam_properties(dataset, dev_motor, droi, threshold)

    for sc in beam_propties.keys():
        xval   = beam_propties[sc][0]
        cx, cy = beam_propties[sc][1]

        centroids[sc] = [xval, [cx, cy]]
 
    return centroids


def beam_fwhm(dataset, dev_motor, droi=4, threshold=THRESHOLD):
    """Return fwhm of the beam profiles from the data dict.

    Args:
        dataset (dict): Nested dict containing the set of one full scan
            (one pass).
        dev_motor (str): The device and motor being scanned (e.g., 'sample.x').
        droi (int): The half-size of the region around the centroid used to
            determine if the beam is visible. Default is 4 pixels.
        threshold (float): The minimum peak-to-average ratio required to
            consider the beam visible. Default is 10.

    Returns:
        dict: A dict mapping scan numbers to (fx, fy) fwhm's.
    """
    fwhms = {}
    _, beam_propties = beam_properties(dataset, dev_motor, droi, threshold)

    for sc in beam_propties.keys():
        xval   = beam_propties[sc][0]
        fx, fy = beam_propties[sc][2]

        fwhms[sc] = [xval, [fx, fy]]

    return fwhms


def beam_intensity(dataset, dev_motor, droi=4, threshold=THRESHOLD):
    """Return the total intensity of the beam profiles from the data dict.

    Args:
        dataset (dict): Nested dict containing the set of one full scan
            (one pass).
        dev_motor (str): The device and motor being scanned (e.g., 'sample.x').
        droi (int): The half-size of the region around the centroid used to
            determine if the beam is visible. Default is 4 pixels.
        threshold (float): The minimum peak-to-average ratio required to
            consider the beam visible. Default is 10.

    Returns:
        dict: A dict mapping scan numbers to total intensity.
    """
    beam_images, beam_propties = beam_properties(dataset, dev_motor,
                                                 droi, threshold)
    exptime = dataset['scan-0000']['dvf_B1']['attrs']['expo_time']

    intensities = {}
    droi = 2
    for sc in beam_images.keys():
        img            = beam_images[sc]
        xval           = beam_propties[sc][0]
        cx, cy         = beam_propties[sc][1]
        fwhm_x, fwhm_y = beam_propties[sc][2]

        peak = np.mean(img[cy - droi : cy + droi + 1,
                           cx - droi : cx + droi + 1])

        mask = img > (peak / 2)
        area_mask  = np.sum(mask)
        img_masked = np.where(mask, img, 0)
        area_img_masked = np.sum(img_masked)

        intensity_by_mask = (area_img_masked / (area_mask * exptime)
                             if area_mask != 0 else 0)

        peak /= exptime
        peak_fwhm_norm = (peak / (fwhm_x * fwhm_y)
                          if fwhm_x * fwhm_y != 0 else 0)

        intensities[sc] = [xval, [peak, intensity_by_mask, peak_fwhm_norm]]

    return intensities


def observable_data(data, observable):
    """Extract the behavior of a variable across scans in a dataset.

    Args:
        data (dict): Nested dict containing the set of
            one full scan (one pass).
        observable (str): The variable to extract
            (e.g., 'photocollector', 'centroid').

    Returns:
        tuple: (motor, scans, xval, yval) where:
            motor (str)      : The name of the scanned variable.
            scans (np.array) : Array of scan numbers.
            xval (np.array)  : Array of scanned variable values.
            yval (list of np.array): List of arrays containing the observable
                values for each scan.
    """
    # The scanned variable (idx).
    motor     = data['scan-0000']['attrs']['scan_motor']
    device    = data['scan-0000']['attrs']['scan_type']
    dev_motor = f"{device}.{motor}"

    # Centroids are calculated in beam_centroid().
    if observable == 'centroid':
        centroids = beam_centroid(data, dev_motor)

        # Dict is ordered by scan number.
        scans = np.array(list(centroids.keys()))

        # Observable values.
        xvals = np.array([centroids[sc][0] for sc in scans])

        # Cetroid values.
        values = np.array([centroids[sc][1] for sc in scans])
        centr  = [values[:, 0], values[:, 1]]

        return motor, scans, xvals, centr

    # FWHMs are calculated in beam_fwhm().
    if observable == 'fwhm':
        fwhms = beam_fwhm(data, dev_motor)

        # Dict is ordered by scan number.
        scans = np.array(list(fwhms.keys()))

        # Observable values.
        xvals = np.array([fwhms[sc][0] for sc in scans])

        # Cetroid values.
        values = np.array([fwhms[sc][1] for sc in scans])
        fs     = [values[:, 0], values[:, 1]]

        return motor, scans, xvals, fs

    # Intensities are calculated in beam_intensity().
    if observable == 'intensity':
        intensities = beam_intensity(data, dev_motor)

        # Dict is ordered by scan number.
        scans = np.array(list(intensities.keys()))

        # Observable values.
        xvals = np.array([intensities[sc][0] for sc in scans])

        # Cetroid values.
        values      = np.array([intensities[sc][1] for sc in scans])
        intensities = [values[:, 0], values[:, 1], values[:, 2]]

        return motor, scans, xvals, intensities

    scans, xval, yval = [], [], []
    # Run over each point scanned.
    for scan, scandata in data.items():
        # Get scan number, scanning index and observable value.
        scans.append(int(scan.split('-')[-1]))
        xmeta = scandata['attrs'].get(dev_motor)
        ymeta = scandata['attrs'].get(f"{device}.{observable}")

        # DEBUG
        # print(f">>> OBSERVABLE DATA - "
        #       f"Scan {scan} / : {device}.{observable}, "
        #       f" xmeta={xmeta} ({type(xmeta)}),"
        #       f" ymeta={ymeta} ({type(ymeta)}) <<<<")
        # DEBUG

        # Append the values, handling both scalar and array metadata cases.
        if isinstance(xmeta, list) or isinstance(xmeta, np.ndarray):
            xval.append(float(xmeta[0]))
        else:
            xval.append(float(xmeta))
        if isinstance(ymeta, list) or isinstance(ymeta, np.ndarray):
            yval.append(float(ymeta[0]))
        else:
            yval.append(float(ymeta))

    return motor, np.array(scans), np.array(xval), [np.array(yval)]


def dataset_plot(ax, xvals, yvals, datakey, observable, motor,
                 first_item=0, last_item=None, annotate_points=True):
    """Plot the behavior of an observable across scans for a single dataset."""
    # Plot the observable vs. scan number for the given dataset.
    # for yval in yvals:

    if annotate_points:
        for i, (tx, yv) in enumerate(zip(xvals[first_item:last_item],
                                         yvals[first_item:last_item])):
            ax.annotate(str(i), (tx, yv), textcoords='offset points',
                         xytext=(5, 5), fontsize=8)
    if last_item is None:
        ax.plot(xvals[first_item:], yvals[first_item:],
                marker='o', label=f'Dataset {datakey}')
    else:
        ax.plot(xvals[first_item:last_item],
                yvals[first_item:last_item],
                marker='o', label=f'Dataset {datakey}')

    ax.set_xlabel(motor)
    ax.set_ylabel(observable)
    ax.set_title(f'{observable.capitalize()} vs. {motor.capitalize()}')
    ax.legend()
    ax.grid(True)


def plot_double_observable(axs, nrow, data, observable, observables,
                           first_item=0, last_item=None):
    """Plot two-component observables in separate subplots."""
    for datakey, dataset in data.items():
        motor, scans, xvals, yvals = observable_data(dataset, observable)
        dataset_plot(axs[nrow, 0], xvals, yvals[0], datakey,
                     f"{observable} X", motor, first_item, last_item)
        dataset_plot(axs[nrow, 1], xvals, yvals[1], datakey,
                     f'{observable} Y', motor, first_item, last_item)
    observables.remove(observable)
    return


def scan_plot(data, observables, first_item=0, last_item=None):
    """Plot the behavior of an observable across scans for each dataset.

    Args:
        data (dict): Nested dict containing the set of all passes.
        observables (list of str): The variables to plot
            (e.g., 'photocollector', 'centroid').
        first_item (int): The first point of the scan to be included in
            the plot. It allows skipping initial points if desired.
            Default is 0 (include all points).
        last_item (int or None): The last point of the scan to be included
            in the plot. If None, include all points.
    """
    # Determine the number of rows and columns for subplots based on
    # the number of observables.
    nobs = len(observables)
    # Add an extra plot for the centroid components.
    nobs += sum([1 for obs in ['centroid', 'fwhm', 'intensity']
                 if obs in observables])

    nrows = max((nobs + 1) // 2, 1)
    ncols = 2 if nobs > 1 else 1
    fig, axs = plt.subplots(nrows=nrows, ncols=ncols,
                            figsize=(10 * ncols, 6 * nrows))
    # Make it iterable
    if nrows == 1 and ncols == 1:
        axs = [axs]
    # elif nrows == 1 or ncols == 1:
    #     axs = axs.flatten()

    # Some observables demand two subplots.
    nextrow = 0
    for observable in ['centroid', 'fwhm', 'intensity']:
        if observable in observables:
            plot_double_observable(axs, nextrow, data,
                                   observable, observables,
                                   first_item, last_item)
            nextrow += 1

    # Loop over each observable and dataset to plot the
    # observable vs. scan number.
    for idx, observable in enumerate(observables):
        nr, nc = divmod(idx + nextrow * ncols, 2)
        ax = axs[nr, nc] if nrows > 1 and ncols > 1 else axs[idx + nextrow]
        for datakey, dataset in data.items():
            motor, scans, xvals, yvals = observable_data(dataset, observable)
            for yval in yvals:
                dataset_plot(ax, xvals, yval, datakey, observable, motor,
                            first_item, last_item)

    plt.show()


def centroid_plot(dataset, scanpass, wdir='.', save_fmt='gif'):
    """Plot the beam profile images and their centroids in a dataset.

    Args:
        dataset (dict): Nested dict containing the set of one full scan
            (one pass).
        scanpass (str): Identifier for the scan pass
            (used in titles and filenames).
        wdir (str): Working directory to save the plots.
            Default is current directory.
        save_fmt (str): Format to save the animation
            ('gif', 'mp4', or '' to skip saving).
    """
    # Use the existing function to get motor, positions and centroids.
    (motor, scan_nums, xval,
     (cx_all, cy_all)) = observable_data(dataset, 'centroid')

    # Retrieve images in the same order as scan_nums.
    images = [(f'scan-{sc:04d}',
            dataset[f'scan-{sc:04d}']['dvf_B1']['data'])
            for sc in scan_nums]

    fig, (ax_img, ax_cx, ax_cy) = plt.subplots(1, 3, figsize=(24, 5))

    # Left: image
    im = ax_img.imshow(images[0][1], cmap='viridis', animated=True)
    plt.colorbar(im, ax=ax_img, label='Intensity')
    ax_img.set_xlabel('Pixel X')
    ax_img.set_ylabel('Pixel Y')
    img_title = ax_img.set_title('')

    # Center: centroid X trace with a moving marker
    ax_cx.plot(xval, cx_all, marker='o',
            color='steelblue', label='Centroid X')
    marker_pt, = ax_cx.plot([], [], 'ro',
                            markersize=10, label='Current')
    ax_cx.set_xlabel(motor.capitalize())
    ax_cx.set_ylabel('Centroid X (px)')
    ax_cx.set_title('Centroid X')
    ax_cx.legend()
    ax_cx.grid(True)

    # Right: centroid Y trace with a moving marker
    ax_cy.plot(xval, cy_all, marker='o',
               color='orange', label='Centroid Y')
    marker_pt_y, = ax_cy.plot([], [], 'ro',
                              markersize=10, label='Current')
    ax_cy.set_xlabel(motor.capitalize())
    ax_cy.set_ylabel('Centroid Y (px)')
    ax_cy.set_title('Centroid Y')
    ax_cy.legend()
    ax_cy.grid(True)

    def update(frame):
        name, img = images[frame]
        im.set_data(img)
        im.set_clim(img.min(), img.max())
        img_title.set_text(f'Scan: {name}')
        marker_pt.set_data([xval[frame]], [cx_all[frame]])
        marker_pt_y.set_data([xval[frame]], [cy_all[frame]])
        return im, img_title, marker_pt, marker_pt_y

    anim = FuncAnimation(fig, update, frames=len(images),
                         interval=300, blit=True)
    plt.tight_layout(pad=3.0)

    if save_fmt == 'gif':
        anim.save(f'{wdir}/beam_xy_pass_{scanpass}.gif',
                writer='pillow', fps=2, dpi=300)
    elif save_fmt == 'mp4':
        # Alternatively, save as mp4 (requires ffmpeg):
        anim.save(f'{wdir}/beam_xy_pass_{scanpass}.mp4',
                writer='ffmpeg', fps=2, dpi=300)
    else:
        print("File format not specified or unknown"
              " (use 'gif' or 'mp4' to save to file)."
              " Skipping saving process.")
        pass

    plt.close()
    ipydisplay(HTML(anim.to_jshtml()))


def fwhm_plot(dataset, scanpass, wdir='.', save_fmt='gif'):
    """Plot the beam profile images and their FWHMs in a dataset.

    Args:
        dataset (dict): Nested dict containing the set of one full scan
            (one pass).
        scanpass (str): Identifier for the scan pass
            (used in titles and filenames).
        wdir (str): Working directory to save the plots.
            Default is current directory.
        save_fmt (str): Format to save the animation
            ('gif', 'mp4', or '' to skip saving).
    """
    # Use the existing function to get motor, positions and centroids.
    (motor, scan_nums, xval,
     (fx_all, fy_all)) = observable_data(dataset, 'fwhm')

    # Retrieve images in the same order as scan_nums.
    images = [(f'scan-{sc:04d}',
            dataset[f'scan-{sc:04d}']['dvf_B1']['data'])
            for sc in scan_nums]

    fig, (ax_img, ax_fx, ax_fy) = plt.subplots(1, 3, figsize=(24, 5))

    # Left: image
    im = ax_img.imshow(images[0][1], cmap='viridis', animated=True)
    plt.colorbar(im, ax=ax_img, label='Intensity')
    ax_img.set_xlabel('Pixel X')
    ax_img.set_ylabel('Pixel Y')
    img_title = ax_img.set_title('')

    # Center: centroid X trace with a moving marker
    ax_fx.plot(xval, fx_all, marker='o',
            color='steelblue', label='FWHM X')
    marker_pt, = ax_fx.plot([], [], 'ro',
                            markersize=10, label='Current')
    ax_fx.set_xlabel(motor.capitalize())
    ax_fx.set_ylabel('FWHM X (px)')
    ax_fx.set_title('FWHM X')
    ax_fx.legend()
    ax_fx.grid(True)

    # Right: centroid Y trace with a moving marker
    ax_fy.plot(xval, fy_all, marker='o',
               color='orange', label='FWHM Y')
    marker_pt_y, = ax_fy.plot([], [], 'ro',
                              markersize=10, label='Current')
    ax_fy.set_xlabel(motor.capitalize())
    ax_fy.set_ylabel('FWHM Y (px)')
    ax_fy.set_title('FWHM Y')
    ax_fy.legend()
    ax_fy.grid(True)

    def update(frame):
        name, img = images[frame]
        im.set_data(img)
        im.set_clim(img.min(), img.max())
        img_title.set_text(f'Scan: {name}')
        marker_pt.set_data([xval[frame]], [fx_all[frame]])
        marker_pt_y.set_data([xval[frame]], [fy_all[frame]])
        return im, img_title, marker_pt, marker_pt_y

    anim = FuncAnimation(fig, update, frames=len(images),
                         interval=300, blit=True)
    plt.tight_layout(pad=3.0)

    if save_fmt == 'gif':
        anim.save(f'{wdir}/beam_xy_pass_{scanpass}_fwhm.gif',
                writer='pillow', fps=2, dpi=300)
    elif save_fmt == 'mp4':
        # Alternatively, save as mp4 (requires ffmpeg):
        anim.save(f'{wdir}/beam_xy_pass_{scanpass}_fwhm.mp4',
                writer='ffmpeg', fps=2, dpi=300)
    else:
        print("File format not specified or unknown"
              " (use 'gif' or 'mp4' to save to file)."
              " Skipping saving process.")
        pass

    plt.close()
    ipydisplay(HTML(anim.to_jshtml()))


def observable_statistics(data, observable):
    """Calculate statistics of an observable across scans in a dataset.

    Args:
        data (dict): Nested dict containing the set of all passes.
        observable (str): The variable to analyze (e.g., 'photocollector',
            'centroid').
        droi (int): The half-size of the region around the centroid used to
            determine if the beam is visible. Default is 4 pixels.
        threshold (float): The minimum peak-to-average ratio required to
            consider the beam visible. Default is 10.

    Returns:
        dict: A dict mapping dataset keys to statistics of the observable,
            including mean, median, and standard deviation.
    """
    yscans = []
    for datakey, dataset in data.items():
        motor, scans, xval, yvals = observable_data(dataset, observable)
        yscans.append(yvals)

    # Calculate statistics for each dataset.
    yscans = np.array(yscans)
    yavg   = np.mean(yscans, axis=0)
    ymed   = np.median(yscans, axis=0)
    ystd   = np.std(yscans, axis=0)

    stats = {
        'xval'    : xval,
        'mean'    : yavg,
        'median'  : ymed,
        'std_dev' : ystd
    }

    return stats


### Main entry for data reading.

In [ ]:
# workdir = "/ibira/lnls/beamlines/carcara/Carcara-X/data"
workdir = "/home/arnaldo.filho/Carcara/measurement/2026-03-19"
file_pattern = r"mirror_tx_pass[01][0-9]*"


def main(workdir, file_pattern):
    """Main function to load data and generate plots."""
    files = files_in_directory(workdir, file_pattern)
    data  = data_from_h5_files(files)
    print(f"Loaded {len(files)} files matching pattern "
          f"'{file_pattern}' from '{workdir}'\n Files:")

    for f in files:
        print(f" * {f}")
    return data


if __name__ == "__main__":

    # observable = 'centroid'
    # observable = 'photocollector'
    observable = 'tx'
    pass_interval = (0, 9)
    item_interval = (0, None)
    data = main(workdir, file_pattern)

#### Evaluate the differences between first and last position.
Check rotation Rz due to Tx movement.
Compute accumulated variation through scans.

In [ ]:
def plot_centroid_x_delta(data, motor, scan_start=0, scan_end=12):
    """Plot motor change and centroid X change across scan passes.

    Args:
        data (dict): Nested dict containing all scan passes.
        motor (str): Motor observable to plot, e.g. 'mirror.cs_rz' or 'mirror.ry'.
        scan_start (int): Scan index for the initial centroid measurement.
        scan_end (int): Scan index for the final centroid measurement.
    """
    fig, axs = plt.subplots(2, 1, figsize=(10, 10))
    rax0 = axs[0].twinx()
    rax1 = axs[1].twinx()
    plt.subplots_adjust(hspace=0.3)

    baseline_motor = []
    final_motor    = []
    scan_idx       = []
    baseline_cx    = []
    final_cx       = []

    for datakey, dataset in data.items():
        baseline_motor.append(dataset[f'scan-{scan_start:04d}']['attrs'][motor][0])
        final_motor.append(dataset[f'scan-{scan_end:04d}']['attrs'][motor][0])
        scan_idx.append(len(scan_idx))

        cx = beam_centroid(dataset, 'mirror.tx')
        baseline_cx.append(cx[scan_start][1][0])
        final_cx.append(cx[scan_end][1][0])

    init_motor = baseline_motor[0]
    init_cx    = baseline_cx[0]

    baseline_motor = np.array(baseline_motor) - init_motor
    final_motor    = np.array(final_motor) - init_motor
    baseline_cx    = np.array(baseline_cx) - init_cx
    final_cx       = np.array(final_cx) - init_cx

    motor_delta = final_motor - baseline_motor
    cx_delta    = final_cx - baseline_cx

    motor_cumulative = np.cumsum(motor_delta)
    cx_cumulative    = np.cumsum(cx_delta)

    motor_label = motor.replace('.', ' ').upper()
    title = f'{motor_label} Change vs. Scan Index'
    left_label = f'{motor_label} Change'

    line0,  = axs[0].plot(scan_idx, motor_delta, marker='o', color='blue',
                          label=motor_label)
    line0b, = rax0.plot(scan_idx, cx_delta, marker='o', color='green',
                       label='Centroid X Change (px)')

    lines, labels = axs[0].get_legend_handles_labels()
    lines2, labels2 = rax0.get_legend_handles_labels()
    axs[0].legend(lines + lines2, labels + labels2, loc='best')

    axs[0].grid(True)
    axs[0].set_xlabel('Scan Index')
    axs[0].set_ylabel(left_label)
    rax0.set_ylabel('Centroid X Change (px)')
    axs[0].set_title(title)

    axs[1].plot(scan_idx, motor_cumulative, marker='o', color='blue',
                label=f'Cumulative {motor_label} Change')
    rax1.plot(scan_idx, cx_cumulative, marker='o', color='green',
              label='Cumulative Centroid X Change')

    lines, labels = axs[1].get_legend_handles_labels()
    lines2, labels2 = rax1.get_legend_handles_labels()
    axs[1].legend(lines + lines2, labels + labels2, loc='best')

    axs[1].grid(True)
    axs[1].set_xlabel('Scan Index')
    axs[1].set_ylabel(f'Cumulative {motor_label} Change')
    rax1.set_ylabel('Cumulative Centroid X Change (px)')
    axs[1].set_title(f'Cumulative {motor_label} Change vs. Scan Index')

    plt.show()


In [ ]:
motors = ['mirror.tx',
          'mirror.ry',
          'mirror.y1',
          'mirror.y2',
          'mirror.y3',
          'mirror.cs_tx',
          'mirror.cs_ty',
          'mirror.cs_rx',
          'mirror.cs_rz',
          ]

for motor in motors:
    plot_centroid_x_delta(data, motor, scan_start=0, scan_end=12)

### Observable plot for all passes.

In [ ]:
# observables = ['centroid']
observables = [
    'ry', 'y1', 'y2', 'y3',
    'photocollector',
    'centroid',
    'fwhm',
    'intensity',
    'cs_ty',
    'cs_rz',
    'cs_rx',
    ]

pass_interval = (1, 9)  # (0, 9)
item_interval = (0, 13)  # (1, 12)

# Chose set interval of scans to plot.
first_pass, last_pass = pass_interval
# sel_data = data
sel_data = dict(list(data.items())[first_pass:last_pass+1])

fr, upto = item_interval

scan_plot(sel_data, observables, first_item=fr, last_item=upto)

#### Statistics (avg, std dev, median) after a set of scans.

In [ ]:
stats = observable_statistics(data, observable)


#### Functions to extract device (motor) and observable values across scans.

In [ ]:
# Get scanned values from read data.

def _get_dev_val(dataset, nscan, dev):
    """Helper function to extract a device value from the dataset."""
    val = dataset[nscan]['attrs'].get(dev)
    if isinstance(val, (np.ndarray, list)):
        val = val[0]
    return val


def get_scan_data(data, variable, observable):
    """Extract observable and variable values across scans."""
    ndset = list(data.keys())
    datascans = []
    for ns in ndset:
        scanlist = list(data[ns].keys())

        obs_set = []
        var_set = []

        for nscan in scanlist:
            obsval = _get_dev_val(data[ns], nscan, observable)
            obs_set.append(obsval)
            varval = _get_dev_val(data[ns], nscan, variable)
            var_set.append(varval)

        datascans.append((obs_set, var_set))

    return datascans


#### Show centroid following initial tx.

In [ ]:
import matplotlib.cm as cm
import numpy as np

ctx_0 = []
ctx_m = []
ctx_f = []
npti = 0
nptf = 12

for data in data.values():
    bc  = beam_centroid(data, 'mirror.tx')
    tx0 = bc[npti][0]
    cx0 = bc[npti][1]
    ctx_0.append(np.array([tx0, cx0]))

    txm = bc[(npti + nptf) // 2][0]
    cxm = bc[(npti + nptf) // 2][1]
    ctx_m.append(np.array([txm, cxm]))

    txf = bc[nptf][0]
    cxf = bc[nptf][1]
    ctx_f.append(np.array([txf, cxf]))

ctx_0 = np.array(ctx_0)
ctx_m = np.array(ctx_m)
ctx_f = np.array(ctx_f)

n = len(ctx_0)
colors = cm.viridis(np.linspace(0, 1, n))

fig, (axx, axc, axy) = plt.subplots(1, 3, figsize=(24, 6))

for i, (idx, yv) in enumerate(ctx_0):
    axx.scatter(idx, yv[0], color=colors[i], zorder=3, s=60)
    axx.annotate(str(i), (idx, yv[0]), textcoords='offset points',
                 xytext=(5, 5), fontsize=8)

for i, (idx, yv) in enumerate(ctx_m):
    axc.scatter(idx, yv[0], color=colors[i], zorder=3, s=60)
    axc.annotate(str(i), (idx, yv[0]), textcoords='offset points',
                 xytext=(5, 5), fontsize=8)

for i, (idx, yv) in enumerate(ctx_f):
    axy.scatter(idx, yv[0], color=colors[i], zorder=3, s=60)
    axy.annotate(str(i), (idx, yv[0]), textcoords='offset points',
                 xytext=(5, 5), fontsize=8)


sm = plt.cm.ScalarMappable(cmap='viridis',
                            norm=plt.Normalize(vmin=0, vmax=n - 1))
plt.colorbar(sm, ax=axx, label='Sequence index')
plt.colorbar(sm, ax=axc, label='Sequence index')
plt.colorbar(sm, ax=axy, label='Sequence index')

xmin, xmax = np.min(ctx_0[:, 0]) * 1.002, np.max(ctx_0[:, 0]) * 0.998

axx.grid(True)
axx.set_title("Centroid X vs. Tx, Initial position")
axx.set_xlabel('Tx initial (mm)')
axx.set_ylabel('Centroid X (px)')
axx.set_xlim(xmin, xmax)
axx.set_ylim(2400, 2460)

axc.grid(True)
axc.set_title("Centroid X vs. Tx, Intermediate position")
axc.set_xlabel('Tx intermediate (mm)')
axc.set_ylabel('Centroid X (px)')
axc.set_xlim(xmin, xmax)
axc.set_ylim(2470, 2500)
axc.grid(True)

axy.grid(True)
axy.set_title("Centroid X vs. Tx, Final position")
axy.set_xlabel('Tx final (mm)')
axy.set_ylabel('Centroid X (px)')
axy.set_xlim(xmin, xmax)
axy.set_ylim(2400, 2460)

plt.show()

#### Compare centroids at the beginning, middle and end points of each pass.

In [ ]:
import matplotlib.cm as cm
import numpy as np

cty_0 = []
cty_m = []
cty_f = []
npti = 0
nptf = 12

for data in data.values():
    bc  = beam_centroid(data, 'mirror.tx')
    tx0 = bc[npti][0]
    cy0 = bc[npti][1]
    cty_0.append(np.array([tx0, cy0[1]]))

    txm = bc[(npti + nptf) // 2][0]
    cym = bc[(npti + nptf) // 2][1]
    cty_m.append(np.array([txm, cym[1]]))

    txf = bc[nptf][0]
    cyf = bc[nptf][1]
    cty_f.append(np.array([txf, cyf[1]]))

cty_0 = np.array(cty_0)
cty_m = np.array(cty_m)
cty_f = np.array(cty_f)

n = len(cty_0)
colors = cm.viridis(np.linspace(0, 1, n))

fig, (axx, axc, axy) = plt.subplots(1, 3, figsize=(24, 6))

for i, (idx, cy) in enumerate(cty_0):
    axx.scatter(idx, cy, color=colors[i], zorder=3, s=60)
    axx.annotate(str(i), (idx, cy), textcoords='offset points',
                 xytext=(5, 5), fontsize=8)

for i, (idx, cy) in enumerate(cty_m):
    axc.scatter(idx, cy, color=colors[i], zorder=3, s=60)
    axc.annotate(str(i), (idx, cy), textcoords='offset points',
                 xytext=(5, 5), fontsize=8)

for i, (idx, cy) in enumerate(cty_f):
    axy.scatter(idx, cy, color=colors[i], zorder=3, s=60)
    axy.annotate(str(i), (idx, cy), textcoords='offset points',
                 xytext=(5, 5), fontsize=8)


sm = plt.cm.ScalarMappable(cmap='viridis',
                            norm=plt.Normalize(vmin=0, vmax=n - 1))
plt.colorbar(sm, ax=axx, label='Sequence index')
plt.colorbar(sm, ax=axc, label='Sequence index')
plt.colorbar(sm, ax=axy, label='Sequence index')

xmin, xmax = np.min(cty_0[:, 0]) * 1.002, np.max(cty_0[:, 0]) * 0.998
ymin, ymax = np.min(cty_0[:, 1]) * 0.998, np.max(cty_0[:, 1]) * 1.002

axx.grid(True)
axx.set_title("Centroid Y vs. Tx, Initial position")
axx.set_xlabel('Tx initial (mm)')
axx.set_ylabel('Centroid Y (px)')
axx.set_xlim(xmin, xmax)
axx.set_ylim(ymin, ymax)

axc.grid(True)
axc.set_title("Centroid Y vs. Tx, Intermediate position")
axc.set_xlabel('Tx intermediate (mm)')
axc.set_ylabel('Centroid Y (px)')
axc.set_xlim(xmin, xmax)
axc.set_ylim(ymin, ymax)
axc.grid(True)

axy.grid(True)
axy.set_title("Centroid Y vs. Tx, Final position")
axy.set_xlabel('Tx final (mm)')
axy.set_ylabel('Centroid Y (px)')
axy.set_xlim(xmin, xmax)
axy.set_ylim(ymin, ymax)

plt.show()

#### Smoothed curves (savgol filter) to test centroid calculations.

In [ ]:
import numpy as np
# from scipy.signal import savgol_filter

img = data['01']['scan-0012']['dvf_B1']['data']
yv = np.sum(img, axis=0).argmax()
cy = np.sum(img, axis=1).argmax()

# FWHMs
x_profile = img[cy, :]
y_profile = img[:, yv]

halfx = x_profile.max() / 2
halfy = y_profile.max() / 2


xsmooth = savgol_filter(x_profile, window_length=13, polyorder=2)
ysmooth = savgol_filter(y_profile, window_length=13, polyorder=2)

xmaxraw = np.argmax(x_profile)
ymaxraw = np.argmax(y_profile)
xmax = np.argmax(xsmooth)
ymax = np.argmax(ysmooth)

print(f"Raw centroid: ({xmaxraw}, {ymaxraw}),\n"
      f" Smoothed centroid: ({xmax}, {ymax})")

fwhm_x_raw = np.sum(x_profile > halfx)
fwhm_y_raw = np.sum(y_profile > halfy)
fwhm_x = np.sum(xsmooth > halfx)
fwhm_y = np.sum(ysmooth > halfy)

print(f"FWHM X (raw): {fwhm_x_raw} pixels, FWHM Y (raw): {fwhm_y_raw} pixels")
print(f"FWHM X: {fwhm_x} pixels, FWHM Y: {fwhm_y} pixels")


# plt.plot(x_profile[x_profile > halfx], label='x')
# plt.plot(y_profile[y_profile > halfy], label='y')
plt.plot(xsmooth[xsmooth > halfx], label='x smooth')
plt.plot(ysmooth[ysmooth > halfy], label='y smooth')
# plt.plot([0, len(x_profile)],
#          [x_profile.max() / 2, x_profile.max() / 2],
#          'k--', label='Half max')
plt.grid(True)
plt.legend()


#### Mean and meadian of a single curve, for testing.

In [ ]:
idx = stats['xval']
yv = stats['mean'][0]
cy = stats['mean'][1]
dx = stats['std_dev'][0]
dy = stats['std_dev'][1]
mx = stats['median'][0]
my = stats['median'][1]

fig, (axmx, axmy) = plt.subplots(1, 2, figsize=(15, 6))

axmx.errorbar(idx, yv, yerr=dx, fmt='o-', label='Mean ± Std Dev')
axmx.set_ylabel(f'x {observable} Mean')
axmx.set_title(f'{observable} x Mean with Std Dev')

axmy.errorbar(idx, cy, yerr=dy, fmt='o-', label='Mean ± Std Dev')
axmy.set_ylabel(f'y {observable} Mean')
axmy.set_title(f'{observable} y Mean with Std Dev')

axmx.plot(idx, mx, 's-', label='Median', color='red')
# axmx.set_ylabel(f'x {observable} Median')
# axmx.set_title(f'{observable} x Median')

axmy.plot(idx, my, 's-', label='Median', color='red')
# axmy.set_ylabel(f'y {observable} Median')
# axmy.set_title(f'{observable} y Median')

for axs in [axmx, axmy]:
    axs.grid(True)
    axs.set_xlabel('Tx')
    axs.legend()
    axs.grid(True)

### Sequence of beam images.

In [ ]:
# Working directory for saving the plots.
wdir = "/home/ABTLUS/arnaldo.filho/Beam_profiling/beam_profiles"

# Plot centroids and images for all datasets.
for scanpass, data in data.items():
    centroid_plot(data, scanpass, wdir=wdir, save_fmt='')


#### Evaluation of differences of centroid position along experimental passes.

In [ ]:
import numpy as np

pixsize = 0.48  # um/px

initfindiff = []
scandiff = []
for scanpass, data in data.items():
    print(f"\n##### Scan pass: {scanpass}")

    cent = observable_data(data, 'centroid')
    idx = cent[2][1:-1] * pixsize
    yv = cent[3][0][1:-1] * pixsize

    a, b = np.polyfit(idx, yv, 1)
    cxfit = a * idx + b
    c0a = cent[3][0][0] * pixsize
    c0b = cent[3][0][-1] * pixsize
    c0x = yv[len(yv) // 2]

    c0fit = a * idx[len(idx) // 2] + b

    print("Linear fit parameters:"
          f"               a = {a:.4f}, b = {b:.4f}")
    print("Initial and final centroid values:"
          f"   {c0a:.2f} → {c0b:.2f}")
    print("Central centroid value along scan:"
          f"   {c0x:.2f}")

    dinfin = (c0b / c0a - 1) * 100
    cdiff  = (c0x / c0a - 1) * 100
    scandiff.append(cdiff)
    initfindiff.append(dinfin)
    print("Centroid differences, relative to initial position:")
    print(f"\t at center, along scan:   {(c0x - c0a) / c0a * 100 :.2f}%")
    print(f"\t after scan:              {dinfin:.2f}%")
    print(f"\t at center, based on fit: {(c0fit - c0a) / c0a * 100 :.2f}%")

scandiff = np.array(scandiff)
initfindiff = np.array(initfindiff)

#### Correlation analysis.

In [ ]:
def correlate(a, b):
    """Calculate the normalized cross-correlation between two 1D arrays."""
    a = a - np.mean(a)
    b = b - np.mean(b)
    norm = np.std(a) * np.std(b) * len(a)
    return np.correlate(a, b, mode='full') / norm

In [ ]:
ifdiff = np.array(initfindiff)    # - np.mean(initfindiff)
sdiff  = np.array(scandiff)       # - np.mean(scandiff)

corr_mat = np.corrcoef(sdiff, ifdiff)
print("\nCorrelation coefficient between"
      f" initial/final change and scan change: {corr_mat[0, 1]:.4f}")

# cross_correlation = np.correlate(sdiff, ifdiff, mode='same')
cc = correlate(sdiff, ifdiff)
lags = np.arange(-len(sdiff) + 1, len(sdiff))
best_lag = lags[np.argmax(cc)]
print(f"Best correlation: {np.max(cc)}")
print(f"Shift needed: {best_lag} indices")


# print("Cross-correlation between initial/final change and scan change:"
#       f" {cross_correlation}")

fig, axs = plt.subplots(figsize=(10, 6))
sc = np.array(list(data.keys()))
axs.plot(sc, ifdiff, marker='o', label='initial/final change')
axs.plot(sc, sdiff, marker='s', label='scan/initial change')
# ax.plot(sc, cc, marker='^',
#         label='cross-correlation (normalized)')
axs.grid(True)
# ax.set_ylim(0, max(max(ifdiff), max(sdiff)) * 1.2)
axs.set_xlabel('Scan pass')
axs.set_ylabel('Centroid change (%)')
axs.legend(loc='upper center', bbox_to_anchor=(0.85, 0.72))
axs.set_title('Relative change in central centroid position across scan passes')

print(" Initial/final mean and std. deviation values:"
      f" {np.mean(ifdiff):.4f} ± {np.std(ifdiff):.4f}")
print(" Scan/initial mean and std. deviation values:"
      f" {np.mean(sdiff):.4f} ± {np.std(sdiff):.4f}")

plt.show()

In [ ]:
fig, axs = plt.subplots(figsize=(10, 6))
axs.plot(cc, marker='o', label='initial/final change')
axs.grid(True)
axs.set_title("Cross-correlation between initial/final change and scan change")
axs.set_xlabel('Lag')
axs.set_ylabel('Correlation coefficient')
# ax.legend()
plt.show()
print(np.max(cc))

#### Linear fitting to centroid positions along a scanning.

In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplots(figsize=(8, 6))

data = data['01']
cent = observable_data(data, 'centroid')
idx = cent[2][1:-1] * pixsize
yv = cent[3][0][1:-1] * pixsize

a, b = np.polyfit(idx, yv, 1)
cxfit = a * idx + b
c0a = cent[3][0][0] * pixsize
c0b = cent[3][0][-1] * pixsize
# c0x = cx[len(cx) // 2]

c0fit = a * idx[len(idx) // 2] + b

axs.plot(idx, yv, 'o-', label='Centroid X')
axs.plot(idx, cxfit, 'r--', label='Linear Fit')
axs.plot(cent[2][0] * pixsize, c0a, 'gs', label='Initial Centroid')
axs.plot(cent[2][-1] * pixsize, c0b, 'rs', label='Final Centroid')
axs.set_xlabel(r'$Tx$ [mm]')
axs.set_ylabel(r'centroid $x$ [$\mu$m]')
axs.set_title(r'Centroid $x$ vs $Tx$ with linear fit')
axs.legend()
axs.grid(True)

#### Test with a specific scan frame.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

dx = 4
threshold = 10
data = data['03']
scandata = '0004'

img = data[f'scan-{scandata}']['dvf_B1']['data']
centroid = beam_centroid(data, 'mirror.tx')
idx, (yv, cy) = centroid[int(scandata)]

roi_avg = np.mean(img[cy-dx:cy+dx, yv-dx:yv+dx])
mean = np.mean(img)
ratio_rtom = roi_avg / mean

print(f"ROI average: {roi_avg:.2f}, Mean: {mean:.2f},"
      f" Ratio (ROI/Mean): {ratio_rtom:.2f}")

print(idx, yv, cy)
plt.imshow(img)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

img1 = data['03']['scan-0000']['dvf_A1']['data']
img2 = data['03']['scan-0000']['dvf_B1']['data']

yv = np.sum(img2, axis=0).argmax()
cy = np.sum(img2, axis=1).argmax()

ax1.imshow(img1, cmap='viridis')
ax1.set_title('dvf_A1')

ax2.imshow(img2, cmap='viridis')
ax2.set_title('dvf_B1')

print(f"Centroid (dvf_B1): {yv}, {cy}")